# Solution R7 · Warm-up: the first tree and the depth

**Type**: Implement with scikit-learn

## Objective

Train trees of increasing depth, observe how the training accuracy rises up to 1.0 while the test one stalls, and read the **rule at the root** of a small tree.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

data = load_breast_cancer(as_frame=True)
X, y = data.data, (data.target == 0).astype(int)                       # 1 = malignant
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

## Commented solution

In [ ]:
depths = {"1": 1, "2": 2, "3": 3, "5": 5, "no limit": None}
table = pd.DataFrame(index=list(depths), columns=["train", "test", "leaves"], dtype=float)
for name, maximum in depths.items():
    tree = DecisionTreeClassifier(max_depth=maximum, random_state=0).fit(X_tr, y_tr)
    table.loc[name] = [tree.score(X_tr, y_tr), tree.score(X_te, y_te), tree.get_n_leaves()]
table.round(3)

In [ ]:
gap = float(table["train"].iloc[-1] - table["test"].iloc[-1])
gain_beyond_3 = float(table["test"].iloc[3:].max() - table.loc["3", "test"])
print(table.round(3).to_string())
print(f"\ntraining-test gap without depth limit: {gap:.3f} | test gain beyond depth 3: {gain_beyond_3:+.3f}")

In [ ]:
tree2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_tr, y_tr)
print(export_text(tree2, feature_names=list(X.columns)))
root = X.columns[tree2.tree_.feature[0]]
root_threshold = float(tree2.tree_.threshold[0])
print(f"rule at the root: {root} <= {root_threshold:.3f}")

## Check

In [ ]:
with verify():
    check(table["train"].is_monotonic_increasing and table.loc["1", "leaves"] == 2 and table["train"].iloc[-1] == 1.0, "the training accuracy rises with the depth up to 1.0; depth 1 has 2 leaves", "tree.score(X_tr, y_tr)")
    check(gap > 0.07 and table["test"].max() < 0.95, "the overfitting gap is over 0.07 and the test accuracy does not reach 0.95", "table.loc[None, 'train'] - table.loc[None, 'test']")
    check(gain_beyond_3 < 0.01, "beyond depth 3 the test accuracy does not improve (gain below 0.01): more leaves do not help", "table['test'].iloc[3:].max() - table.loc['3', 'test']")
    check(root in X.columns and any(k in root for k in ["concave", "perimeter", "radius", "area"]) and root_threshold > 0, "the rule at the root is on a shape or size measurement of the nucleus (concave points, perimeter, radius or area)", "X.columns[tree2.tree_.feature[0]]")
done()

## What you learned

- **The training accuracy always rises, the test one does not:** from 0.935 to 1.0 on the training set (the tree without limit has 15 leaves and memorizes), while the test accuracy sits between 0.86 and 0.92 and does not follow the training one. The overfitting gap of the tree without limit is about 0.09.
- **A few rules are enough:** the test accuracy is better with depth 3 (7 leaves, 0.918) than with the tree without limit (15 leaves, 0.906). The first question the tree asks is `mean concave points <= 0.05`: a tumor with many concave points on the contour tends to be malignant. A rule a doctor can read (with another version of scikit-learn the root may be a similar measurement, like `worst perimeter`).
- **The test set here is small** (171 examples): differences of one or two points between depths are in the noise. To choose the depth you use cross-validation (module 7, exercise B1), not the test set.

## Common pitfall

Choosing the depth by looking at the training accuracy: it always rises. Neither the deepest tree nor the "best on the test set" should be used to choose: you need a separate validation.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why does a tree with no depth limit have accuracy 1.0 on the training set?</summary>

Because it keeps splitting until every leaf is pure: it memorizes the training set. It has low bias and high variance, and on the test set it generalizes like a much smaller tree. It is controlled with the depth, the minimum leaf size or cost-complexity pruning.

</details>